#Classificando os tipos de áudio em 3 grupos

1.   Saudáveis
2.   Benignos
3.   Maligno






#Relação Doença x Grupo
https://docs.google.com/document/d/1ZNcoSK70A2wKLze7kgoM-NAEsScekw41FRSPTMEgdNY/edit?usp=sharing

#Importando bibliotecas

In [8]:
import os
import glob
import pandas as pd
import numpy as np
import parselmouth
from parselmouth.praat import call
from datetime import datetime
import warnings

# Ignorar avisos de conversão de data/futuros do pandas para limpar a saída
warnings.filterwarnings('ignore')

print("Bibliotecas importadas com sucesso!")

Bibliotecas importadas com sucesso!


#Função de Extração de Features (Parselmouth)

In [9]:
def extrair_features_voz(caminho_arquivo):
    """
    Lê um arquivo .wav e extrai características acústicas usando Parselmouth.
    """
    try:
        sound = parselmouth.Sound(caminho_arquivo)

        # 1. Extrair Pitch (Frequência Fundamental - F0)
        # O range 75-600Hz cobre vozes graves (homens) e agudas (mulheres/crianças)
        pitch = sound.to_pitch(time_step=None, pitch_floor=75.0, pitch_ceiling=600.0)

        # Se for silêncio absoluto, retorna None
        if pitch.count_voiced_frames() == 0:
            return None

        # Métricas de Frequência (F0)
        f0_mean = call(pitch, "Get mean", 0, 0, "Hertz")
        f0_std  = call(pitch, "Get standard deviation", 0, 0, "Hertz")
        f0_min  = call(pitch, "Get minimum", 0, 0, "Hertz", "Parabolic")
        f0_max  = call(pitch, "Get maximum", 0, 0, "Hertz", "Parabolic")

        # 2. Pulsos (necessário para perturbações de ciclo a ciclo)
        point_process = call([sound, pitch], "To PointProcess (cc)")

        # --- JITTER (Instabilidade na Frequência) ---
        jitter_local = call(point_process, "Get jitter (local)", 0, 0, 0.0001, 0.02, 1.3)
        jitter_rap   = call(point_process, "Get jitter (rap)", 0, 0, 0.0001, 0.02, 1.3)

        # --- SHIMMER (Instabilidade na Amplitude) ---
        shimmer_local = call([sound, point_process], "Get shimmer (local)", 0, 0, 0.0001, 0.02, 1.3, 1.6)
        shimmer_db    = call([sound, point_process], "Get shimmer (local_dB)", 0, 0, 0.0001, 0.02, 1.3, 1.6)

        # --- HNR (Harmonics-to-Noise Ratio - Ruído/Soprosidade) ---
        harmonicity = sound.to_harmonicity_cc(time_step=0.01, minimum_pitch=75.0, silence_threshold=0.1, periods_per_window=1.0)
        hnr = call(harmonicity, "Get mean", 0, 0)

        return {
            "f0_mean": f0_mean,
            "f0_std": f0_std,
            "f0_min": f0_min,
            "f0_max": f0_max,
            "jitter_local": jitter_local * 100, # %
            "jitter_rap": jitter_rap * 100,     # %
            "shimmer_local": shimmer_local * 100, # %
            "shimmer_db": shimmer_db,
            "hnr": hnr
        }
    except Exception as e:
        print(f"❌ Erro no arquivo {os.path.basename(caminho_arquivo)}: {e}")
        return None

print("✅ Função de extração definida!")

✅ Função de extração definida!


#Configuração e Classificação

In [10]:
# --- CONFIGURAÇÃO ---
# Coloque aqui o caminho onde estão as pastas (ex: C:\Dados\INCA)
CAMINHO_RAIZ_AUDIOS = r"C:\Users\Administrador\Desktop\TCC\Código\CriadorCSV\Unificado"

# --- MAPEAMENTO DE GRUPOS ---
# 0 = Saudável
# 1 = Disfonia Orgânica Benigna (O "Confundidor")
# 2 = Câncer / Lesão Pré-Maligna (O Alvo)

classificacao_doencas = {
    # GRUPO 2: ALVO (Câncer e Pré-Câncer)
    "CarcinomaDePregasVocais": 2,
    "TumorDaLaringe": 2,
    "CarcinomaInSitu": 2,
    "DisfoniaDisplasica": 2,
    "LaringeDisplasica": 2,
    "Leucoplasia": 2,
    "Condroma": 2,

    # GRUPO 1: BENIGNO / ORGÂNICO (Patologias que NÃO são câncer)
    "Cisto_CistoRetencao": 1,
    "Disartrofonia_Disartria": 1,
    "DisfoniaEspasmodica": 1,
    "DisturbioMotorLaringeoCentral": 1,
    "EdemaDeReinke": 1,
    "EsclorioseLateralAmiotrofica": 1,
    "Fibroma": 1,
    "Granuloma": 1,
    "GranulomaDeIntubacao": 1,
    "Laringite": 1,
    "LesaoNeuralgia": 1,
    "Monocordite": 1,
    "NodulosVocais": 1,
    "Papiloma": 1,
    "PaquidermiaDeContato": 1,
    "ParalisiaBulba": 1,
    "ParalisiaDoNervoRecorrente": 1,
    "Parkinson": 1,
    "Presbifonia": 1,
    "RefluxoGastroesofagico": 1,
    "ResseccaoParcialFrontolateral": 1,
    "Sinequia": 1,
    "TraumaDeIntubacao": 1,

    # GRUPO 0: SAUDÁVEL (Controle)
    "healthy": 0
}

# Lista das pastas (baseada nas chaves do dicionário acima)
pastas_para_processar = list(classificacao_doencas.keys())

print("✅ Classificação configurada!")
print(f"Total de tipos de patologias listadas: {len(pastas_para_processar)}")

✅ Classificação configurada!
Total de tipos de patologias listadas: 31


#Processamento

In [13]:
# --- BLOCO 4: LOOP DE PROCESSAMENTO (CORRIGIDO PARA DATAS ISO E ALEMÃS) ---

dados_completos = []

print("🚀 Iniciando processamento... (Isso pode demorar)")

# 1. Função auxiliar para ler CSV de vários formatos
def ler_csv_blindado(caminho):
    separadores = [';', ',', '\t']
    encodings = ['utf-8', 'latin1', 'ISO-8859-1', 'cp1252']
    for enc in encodings:
        for sep in separadores:
            try:
                df = pd.read_csv(caminho, sep=sep, encoding=enc, on_bad_lines='skip')
                if len(df.columns) > 1 and len(df) > 0:
                    return df
            except:
                continue
    return pd.DataFrame()

# 2. Função auxiliar inteligente para converter datas
def converter_data_robusta(series):
    # Tenta primeiro o formato ISO (AAAA-MM-DD) que é o que está no seu arquivo
    datas = pd.to_datetime(series, errors='coerce')
    
    # Se ainda tiver datas falhas (NaT), tenta o formato dia primeiro (DD.MM.AAAA)
    mask_falha = datas.isna() & series.notna()
    if mask_falha.any():
        datas[mask_falha] = pd.to_datetime(series[mask_falha], dayfirst=True, errors='coerce')
    
    return datas

# --- LOOP PRINCIPAL ---
for pasta in pastas_para_processar:
    path_pasta = os.path.join(CAMINHO_RAIZ_AUDIOS, pasta)
    path_csv_local = os.path.join(path_pasta, "overview.csv")
    target_grupo = classificacao_doencas.get(pasta)
    
    if not os.path.exists(path_pasta):
        print(f"⚠️ Pasta não encontrada: {pasta}. Pulando...")
        continue

    # Carrega CSV
    df_local = pd.DataFrame()
    if os.path.exists(path_csv_local):
        df_local = ler_csv_blindado(path_csv_local)
        
        if not df_local.empty:
            df_local.columns = df_local.columns.str.strip()
            
            # Garante que ID da gravação é texto para bater com o nome do arquivo
            if 'AufnahmeID' in df_local.columns:
                df_local['AufnahmeID'] = df_local['AufnahmeID'].astype(str)
            
            # --- CORREÇÃO DAS DATAS AQUI ---
            if 'AufnahmeDatum' in df_local.columns:
                df_local['AufnahmeDatum'] = converter_data_robusta(df_local['AufnahmeDatum'])
            if 'Geburtsdatum' in df_local.columns:
                df_local['Geburtsdatum']  = converter_data_robusta(df_local['Geburtsdatum'])
    else:
        print(f"⚠️ 'overview.csv' ausente em {pasta}.")

    # Lista áudios
    audios = glob.glob(os.path.join(path_pasta, "*.wav"))
    print(f"📂 {pasta} [Grupo {target_grupo}]: {len(audios)} áudios encontrados.")

    # Processa áudios
    for audio_path in audios:
        nome_arquivo = os.path.basename(audio_path)
        
        # Extrai ID da Gravação (AufnahmeID)
        try:
            id_gravacao = nome_arquivo.split('-')[0].split('_')[0]
        except:
            id_gravacao = "desconhecido"

        features = extrair_features_voz(audio_path)
        
        if features:
            idade = None
            sexo = None
            id_paciente_real = None 
            
            if not df_local.empty and 'AufnahmeID' in df_local.columns:
                match = df_local[df_local['AufnahmeID'] == id_gravacao]
                
                if not match.empty:
                    dados_row = match.iloc[0]
                    sexo = dados_row.get('Geschlecht')
                    id_paciente_real = dados_row.get('SprecherID')
                    
                    # Cálculo de Idade
                    nasc = dados_row.get('Geburtsdatum')
                    adm = dados_row.get('AufnahmeDatum')
                    
                    if pd.notnull(nasc) and pd.notnull(adm):
                        try:
                            # Calcula idade em anos
                            idade = round((adm - nasc).days / 365.25, 1)
                        except:
                            pass

            linha = {
                "Nome_Arquivo": nome_arquivo,
                "Patologia_Original": pasta,
                "Grupo_Alvo": target_grupo,
                "ID_Gravacao": id_gravacao,
                "ID_Paciente": id_paciente_real,
                "Idade": idade,
                "Sexo": sexo,
                **features
            }
            dados_completos.append(linha)

print("\n🏁 Processamento concluído!")

🚀 Iniciando processamento... (Isso pode demorar)
📂 CarcinomaDePregasVocais [Grupo 2]: 22 áudios encontrados.
📂 TumorDaLaringe [Grupo 2]: 5 áudios encontrados.
📂 CarcinomaInSitu [Grupo 2]: 1 áudios encontrados.
📂 DisfoniaDisplasica [Grupo 2]: 1 áudios encontrados.
📂 LaringeDisplasica [Grupo 2]: 1 áudios encontrados.
📂 Leucoplasia [Grupo 2]: 41 áudios encontrados.
📂 Condroma [Grupo 2]: 1 áudios encontrados.
📂 Cisto_CistoRetencao [Grupo 1]: 6 áudios encontrados.
📂 Disartrofonia_Disartria [Grupo 1]: 19 áudios encontrados.
📂 DisfoniaEspasmodica [Grupo 1]: 64 áudios encontrados.
📂 DisturbioMotorLaringeoCentral [Grupo 1]: 14 áudios encontrados.
📂 EdemaDeReinke [Grupo 1]: 68 áudios encontrados.
📂 EsclorioseLateralAmiotrofica [Grupo 1]: 2 áudios encontrados.
📂 Fibroma [Grupo 1]: 2 áudios encontrados.
📂 Granuloma [Grupo 1]: 2 áudios encontrados.
📂 GranulomaDeIntubacao [Grupo 1]: 4 áudios encontrados.
📂 Laringite [Grupo 1]: 140 áudios encontrados.
📂 LesaoNeuralgia [Grupo 1]: 3 áudios encontrados.

#Criando o Arquivo

In [14]:
if len(dados_completos) > 0:
    df_final = pd.DataFrame(dados_completos)
    
    # Limpeza
    df_final = df_final[ (df_final['Idade'].isnull()) | (df_final['Idade'] > 0) ]
    if 'Sexo' in df_final.columns:
        df_final['Sexo'] = df_final['Sexo'].replace({'w': 'F', 'm': 'M', 'W': 'F', 'M': 'M'})

    # Salva no diretório atual do notebook
    caminho_saida = os.path.join(os.getcwd(), "dataset_laringe_final.csv")
    df_final.to_csv(caminho_saida, index=False)
    
    print(f"💾 ARQUIVO SALVO EM: {caminho_saida}")
    display(df_final.head())
else:
    print("Nada para salvar.")

💾 ARQUIVO SALVO EM: c:\Users\Administrador\Desktop\TCC\Código\CriadorCSV\dataset_laringe_final.csv


,Nome_Arquivo,Patologia_Original,Grupo_Alvo,ID_Gravacao,ID_Paciente,Idade,Sexo,f0_mean,f0_std,f0_min,f0_max,jitter_local,jitter_rap,shimmer_local,shimmer_db,hnr
0,1048-a_n.wav,CarcinomaDePregasVocais,2,1048,1544,59.0,M,169.329504,30.270889,79.097319,248.025140,1.452848,0.731192,4.150929,0.471505,20.472904
1,110-a_n.wav,CarcinomaDePregasVocais,2,110,1307,62.9,M,79.493672,2.073100,75.199094,89.197652,1.967100,1.100514,8.656242,0.843980,9.635692
2,1239-a_n.wav,CarcinomaDePregasVocais,2,1239,1625,57.9,M,170.892557,2.404023,164.043981,175.938641,0.998545,0.576863,6.010533,0.549174,12.977379
3,1245-a_n.wav,CarcinomaDePregasVocais,2,1245,1625,57.9,M,210.788520,3.487506,204.760310,220.037832,1.275788,0.795547,10.060927,0.885818,14.845637
4,1247-a_n.wav,CarcinomaDePregasVocais,2,1247,1633,42.6,M,167.619778,6.586444,153.292094,179.840444,2.822467,1.676677,15.635894,1.409448,6.686018
